# Phase 2 · Pilot of the canonical v2 run (50 patients, Model A)

Runs `xfi.run_xfi` on the first 50 test patients and checks it before the full run is launched.

| Check | Pass criterion |
|---|---|
| P1 | clean-image probabilities equal the v1 `Model_Prob` (max abs diff < 1e-3) |
| P2 | Gaussian blur is deterministic in both versions: v2 and v1 XFI agree for blur (Pearson r > 0.99) |
| P3 | output integrity: expected row counts, XFI inside [0, 4/3], localization present for every positive |
| P4 | input image quality (PSNR, SSIM) by perturbation type and level (informative) |
| P5 | Eigen-CAM sign diagnostics (informative) |
| P6 | baselines (informative) |
| P7 | timing and ETA of the full runs |

Gaussian noise and salt-and-pepper are **expected** to differ from v1: v2 draws one 2-D noise field per image (D-10) from a per-patient generator, v1 drew per-channel noise from one global stream.

## Configuration cell

In [ ]:
import sys, os, json, subprocess, time
from pathlib import Path
import numpy as np, pandas as pd, yaml

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
N_PILOT = 50
WORKERS = 16
MODEL = "A"
OUT = ROOT / "output" / "phase2" / f"pilot_{MODEL}"
V1 = ROOT / "output" / "v1_legacy" / "full_experiment_results_xfi.csv"
RESULTS = {}
def record(k, ok, **info):
    RESULTS[k] = {"pass": None if ok is None else bool(ok), **info}; print(f"[{k}] {'PASS' if ok else 'INFO' if ok is None else 'FAIL'}")
    for a, b in info.items(): print(f"   {a}: {b}")
print(ROOT, OUT)

## Run the pilot

Starts `python -m xfi.run_xfi` as a separate process (the worker pool needs a real `__main__`) and streams its log. Takes a few minutes. If `OUT` already holds finished shards they are reused (resume).

In [ ]:
env = dict(os.environ, PYTHONPATH=str(ROOT / "src"))
cmd = [sys.executable, "-m", "xfi.run_xfi", "--model", MODEL, "--limit", str(N_PILOT), "--shard", "25",
       "--workers", str(WORKERS), "--save-maps", "8", "--out", str(OUT.relative_to(ROOT))]
t0 = time.time()
proc = subprocess.Popen(cmd, cwd=ROOT, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    if "[xfi" in line or "Error" in line or "Traceback" in line: print(line.rstrip())
proc.wait(); print("exit code", proc.returncode, "| seconds", round(time.time() - t0, 1))
m = pd.read_csv(OUT / "results_main.csv"); b = pd.read_csv(OUT / "results_base.csv")
man = json.loads((OUT / "manifest.json").read_text())
print(m.shape, b.shape, man["groups"])

## P1 · Clean-image probabilities against v1

In [ ]:
v1 = pd.read_csv(V1)
v1p = v1.drop_duplicates("Patient").set_index("Patient")["Model_Prob"]
p2 = m.drop_duplicates("Patient").set_index("Patient")["prob_clean"]
d = (p2 - v1p.reindex(p2.index)).abs()
record("P1_probabilities", d.max() < 1e-3, n=len(d), max_abs_diff=float(d.max()), mean_abs_diff=float(d.mean()))

## P2 · v2 against v1 per perturbation type

Blur has no randomness, so v2 and v1 must agree for Grad-CAM and Eigen-CAM (small differences come from batched CAM evaluation). For Gaussian noise and salt-and-pepper only the means are compared.

In [ ]:
key = ["Patient", "Method", "Type", "Level"]
mm = m[m.Method.isin(["GradCAM", "EigenCAM"])].merge(v1[key + ["xfi"]], on=key, suffixes=("_v2", "_v1"))
rows = []
for (meth, t), g in mm.groupby(["Method", "Type"]):
    rows.append(dict(method=meth, type=t, n=len(g), mean_v1=g.xfi_v1.mean(), mean_v2=g.xfi_v2.mean(),
                     pearson=np.corrcoef(g.xfi_v1, g.xfi_v2)[0, 1], max_abs_diff=(g.xfi_v1 - g.xfi_v2).abs().max()))
P2 = pd.DataFrame(rows); print(P2.round(4).to_string(index=False))
blur = P2[P2.type == "Blur"]
record("P2_blur_agreement", (blur.pearson > 0.99).all(), min_pearson=float(blur.pearson.min()), max_abs_diff=float(blur.max_abs_diff.max()))

## P3 · Output integrity

In [ ]:
n_p = m.Patient.nunique(); pos_ids = set(m[m.Label == 1].Patient)
exp_main = n_p * 3 * 9; exp_base = n_p * (9 * 2 + 1 + 2 + 2)
loc_cols = [c for c in m.columns if c.startswith(("iou_", "far_bg", "energy_out", "prec_in"))]
loc_missing = int(m[m.Label == 1][[c for c in loc_cols if not c.startswith("prec_in")]].isna().sum().sum())
in_range = bool(m.xfi.between(-1e-9, 4 / 3 + 1e-9).all() and b.xfi.between(-1e-9, 4 / 3 + 1e-9).all())
record("P3_integrity", len(m) == exp_main and len(b) == exp_base and loc_missing == 0 and in_range,
       rows_main=f"{len(m)} (expected {exp_main})", rows_base=f"{len(b)} (expected {exp_base})",
       positives=len(pos_ids), missing_localization_values=loc_missing, xfi_in_range=in_range,
       degenerate_share_main=m.groupby("Method").degenerate.mean().round(4).to_dict())

## P4 · Input image quality per condition (D-17)

In [ ]:
P4 = m[m.Method == "GradCAM"].groupby(["Type", "Level"])[["input_psnr", "input_ssim"]].mean().round(3)
print(P4.to_string()); record("P4_input_quality", None, note="informative")

## P5 · Eigen-CAM sign diagnostics (D-11)

`eig_minor_lobe`: the ReLU of grad-cam keeps the smaller lobe of the signed projection (the case that sign correction flips). `eig_signed_corr < 0`: the signed projections of the clean and the perturbed image point in opposite directions (a sign flip between them).

In [ ]:
e = m[m.Method == "EigenCAM"]
P5 = e.groupby(["Type", "Level"]).agg(minor_lobe_clean=("eig_minor_lobe_c", "mean"), minor_lobe_noisy=("eig_minor_lobe_n", "mean"),
                                      sign_flip=("eig_signed_corr", lambda s: (s < 0).mean())).round(3)
print(P5.to_string())
cmp = m[m.Method.isin(["EigenCAM", "EigenCAM_SC"])].groupby(["Method", "Type"]).xfi.mean().unstack().round(4)
print(cmp.to_string()); record("P5_eigen_sign", None, minor_lobe_clean_share=float(e.drop_duplicates("Patient").eig_minor_lobe_c.mean()))

## P6 · Baselines (D-14)

In [ ]:
P6 = b.groupby("Method").agg(mean_xfi=("xfi", "mean"), sd=("xfi", "std"), degenerate=("degenerate", "mean"), n=("xfi", "size")).round(4)
print(P6.to_string()); record("P6_baselines", None, note="informative")

## P7 · Timing and ETA of the full runs

In [ ]:
mins = man["minutes"]; per = mins / n_p
eta = per * 5337
record("P7_timing", None, minutes_pilot=mins, seconds_per_patient=round(per * 60, 2),
       eta_full_run_minutes_per_model=round(eta, 1), eta_A_plus_B_hours=round(2 * eta / 60, 2))

## Summary and LOG entry

The full run (`scripts/run_phase3_xfi.sh`) may start only if P1, P2 and P3 pass. Paste this cell's output back.

In [ ]:
from datetime import date
(OUT / "pilot_checks.json").write_text(json.dumps(RESULTS, indent=2, default=str))
gate = all(RESULTS[k]["pass"] for k in ["P1_probabilities", "P2_blur_agreement", "P3_integrity"])
print("GATE (P1, P2, P3):", "PASS" if gate else "FAIL")
print(f"\n## {date.today()} | Phase 2 pilot (Model A, {n_p} patients)")
for k, v in RESULTS.items():
    print(f"- {k}: {'PASS' if v['pass'] else 'INFO' if v['pass'] is None else 'FAIL'} | " + ", ".join(f"{a}={b}" for a, b in v.items() if a != "pass"))